# Z3-Rust-09 : L'enigme d'Einstein (Zebra puzzle)

[← Z3-Python-08 (ordonnancement)](Z3-08-Ordonnancement-Python.ipynb) | [Serie Z3-API](README.md) | [Version Python](Z3-09-Enigme-Einstein-Python.ipynb) | [Z3-Python-10 (cryptarithmes) →](Z3-10-Cryptarithmetic-Python.ipynb)

> **Duree estimee** : 1h30

> **Solveur** : Z3 (crate Rust `z3` 0.23.0) — solveur SMT, execute dans le kernel `rust` (evcxr). Le portage est integral : le meme modele, les memes 15 indices, la meme lecture.

## Le probleme

L'**enigme d'Einstein** (ou *Zebra puzzle*) est un classique de logique : 5 maisons alignees, chacune habitee par une personne de nationalite, couleur, boisson, cigarette et animal differents. On dispose de **15 indices** (le Britannique vit dans la rouge, le Danois boit le the, la verte est a gauche de la blanche, etc.) et la question est : **qui possede le poisson ?**

A la main, la resolution procede par deductions progressives sur un tableau - c'est long et trompeur. L'espace de recherche brut est enorme : chaque attribut est une permutation des 5 maisons, soit $(5!)^5 = 24\,883\,200\,000$ combinaisons. Une **enumeration exhaustive** (brute force) testant les 15 indices pour chaque combinaison prend plusieurs minutes.

C'est typiquement la ou un solveur SMT comme Z3 **fait la difference** : on decrit les 25 variables (position de chaque valeur), le domaine, les 5 contraintes de permutation (`Ast::distinct`) et les 15 indices, et `Solver` trouve l'unique solution en **quelques millisecondes**. Ce notebook porte en Rust l'exemple C# [18_Einsteins_Riddle](../Z3-Linq2Z3/18_Einsteins_Riddle.ipynb) de la serie soeur Z3.Linq.

## Ce que le portage change (et ce qu'il ne change pas)

- **Le solveur** : le crate `z3` (API Rust, liaison native a la bibliotheque Z3) remplace le module Python `z3`. L'API des contraintes est la meme (`assert`, `check`, `get_model`), mais le typage est statique : une variable est un `Int`, une contrainte un `Bool`, et il n'y a plus de `lambda` mais des closures.
- **Les cellules sont autonomes** : sous evcxr, une fonction definie dans une cellule n'est pas garantie dans la suivante. Chaque cellule qui consomme le solveur le **reconstruit** — c'est une contrainte du moteur, pas une propriete du modele.
- **Le modele, lui, ne change pas** : memes variables, memes 15 indices, meme solution, meme tableau affiche.


In [2]:
:dep z3 = "0.23.0"

use z3::full_version;

// La dependance z3 est declaree une seule fois, en tete de carnet :
// un :dep plus bas recompilerait tout et effacerait les variables typees.
println!("z3 version : {}", full_version());

z3 version : 5.3.3.0 430d19e697e2ec9bad79ff210d41b4a7a5fcfa3e 430d19e


## Modelisation : l'encodage par position

L'astuce centrale consiste a **inverser le point de vue**. Au lieu de demander « quelle couleur a la maison *k* ? », on définit pour chaque valeur d'attribut une variable entiere egale a la **position** (numéro de maison 0..4) ou elle se trouve :

- `Brit`, `Swede`, `Dane`, `Norwegian`, `German` : position de chaque nationalite
- `Red`, `Green`, `White`, `Yellow`, `Blue` : position de chaque couleur
- `Tea`, `Coffee`, `Milk`, `Water`, `Beer` : position de chaque boisson
- `PallMall`, `Dunhill`, `Blend`, `BlueMaster`, `Prince` : position de chaque cigarette
- `Dog`, `Bird`, `Cat`, `Horse`, `Fish` : position de chaque animal

Soit **25 variables entieres**. Les contraintes se traduisent alors très naturellement :

1. **Domaine** : chaque variable $\in \{0,1,2,3,4\}$.
2. **Permutation** : dans chaque groupe d'attributs, les 5 positions sont deux a deux distinctes (`Ast::distinct`) - chaque valeur occupe une maison différente.
3. **Les 15 indices** : des egalites de position (`Brit == Red` : le Britannique et la rouge sont dans la même maison) ou des **adjacences** (`|Blend - Cat| == 1` : le fumeur de Blend est voisin du Chat).

In [3]:
use std::collections::HashMap;
use z3::ast::{Ast, Bool, Int};
use z3::Solver;

// Cellule autonome : sous evcxr les fonctions ne sont pas persistees d'une
// cellule a l'autre, le solveur est donc reconstruit ici.
// `skip` : numero (0..14) d'un indice a omettre — sert au bonus « sans un indice ».
fn build_solveur(skip: Option<usize>) -> (Solver, HashMap<String, Int>) {
    let mut s = Solver::new();
    // Les 5 groupes d'attributes, chacun avec ses 5 valeurs.
    let groups: Vec<Vec<&str>> = vec![
        vec!["Brit", "Swede", "Dane", "Norwegian", "German"],
        vec!["Red", "Green", "White", "Yellow", "Blue"],
        vec!["Tea", "Coffee", "Milk", "Water", "Beer"],
        vec!["PallMall", "Dunhill", "Blend", "BlueMaster", "Prince"],
        vec!["Dog", "Bird", "Cat", "Horse", "Fish"],
    ];
    // 25 variables de position (0..4) : une par valeur d'attribut.
    let mut v: HashMap<String, Int> = HashMap::new();
    for grp in &groups {
        for n in grp {
            v.insert(n.to_string(), Int::new_const(n.to_string().as_str()));
        }
    }
    // (1) Domaine 0..4 pour les 25 variables.
    for var in v.values() {
        s.assert(&var.ge(&Int::from_i64(0)));
        s.assert(&var.le(&Int::from_i64(4)));
    }
    // (2) Permutation : les 5 valeurs d'un meme attribut occupent des maisons distinctes.
    for grp in &groups {
        let vars: Vec<Int> = grp.iter().map(|n| v[*n].clone()).collect();
        s.assert(&Int::distinct(&vars));
    }
    // (3) Les 15 indices, dans l'ordre de l'enonce.
    let mut indices: Vec<(&str, Bool)> = Vec::new();
    indices.push(("1", v["Brit"]._eq(&v["Red"])));                        // le Britannique vit dans la rouge
    indices.push(("2", v["Swede"]._eq(&v["Dog"])));                       // le Suedois eleve des chiens
    indices.push(("3", v["Dane"]._eq(&v["Tea"])));                        // le Danois boit le the
    indices.push(("4", v["Green"]._eq(&Int::sub(&[&v["White"], &Int::from_i64(1)])))); // la verte est juste a gauche de la blanche
    indices.push(("5", v["Green"]._eq(&v["Coffee"])));                   // la verte boit le cafe
    indices.push(("6", v["PallMall"]._eq(&v["Bird"])));                  // Pall Mall eleve les oiseaux
    indices.push(("7", v["Yellow"]._eq(&v["Dunhill"])));                 // la jaune fume du Dunhill
    indices.push(("8", v["Milk"]._eq(&Int::from_i64(2))));               // la maison du milieu boit le lait
    indices.push(("9", v["Norwegian"]._eq(&Int::from_i64(0))));          // le Norvegien dans la 1ere maison
    // 10. Blend voisin du Chat
    let d1 = Int::sub(&[&v["Blend"], &v["Cat"]]);
    let d2 = Int::sub(&[&v["Cat"], &v["Blend"]]);
    indices.push(("10", Bool::or(&[&d1._eq(&Int::from_i64(1)), &d2._eq(&Int::from_i64(1))])));
    // 11. Cheval voisin du Dunhill
    let e1 = Int::sub(&[&v["Horse"], &v["Dunhill"]]);
    let e2 = Int::sub(&[&v["Dunhill"], &v["Horse"]]);
    indices.push(("11", Bool::or(&[&e1._eq(&Int::from_i64(1)), &e2._eq(&Int::from_i64(1))])));
    indices.push(("12", v["BlueMaster"]._eq(&v["Beer"])));                // BlueMaster boit de la biere
    indices.push(("13", v["German"]._eq(&v["Prince"])));                  // l'Allemand fume le Prince
    // 14. Norvegien voisin de la bleue
    let f1 = Int::sub(&[&v["Norwegian"], &v["Blue"]]);
    let f2 = Int::sub(&[&v["Blue"], &v["Norwegian"]]);
    indices.push(("14", Bool::or(&[&f1._eq(&Int::from_i64(1)), &f2._eq(&Int::from_i64(1))])));
    // 15. Blend voisin de l'Eau
    let g1 = Int::sub(&[&v["Blend"], &v["Water"]]);
    let g2 = Int::sub(&[&v["Water"], &v["Blend"]]);
    indices.push(("15", Bool::or(&[&g1._eq(&Int::from_i64(1)), &g2._eq(&Int::from_i64(1))])));

    for (i, (_numero, condition)) in indices.iter().enumerate() {
        if Some(i) == skip {
            continue;
        }
        s.assert(condition);
    }
    (s, v)
}

// Cas particulier : l'enigme complete, ses 15 indices.
fn build_einstein_solver() -> (Solver, HashMap<String, Int>) {
    build_solveur(None)
}

println!("Modele encode : 25 variables de position, 5 groupes Distinct, 15 indices.");

Modele encode : 25 variables de position, 5 groupes Distinct, 15 indices.


## Resolution et temoin



On appelle `s.check()` : si `sat`, on extrait le modèle puis on **inverse** l'encodage par position - pour chaque maison $h \in \{0..4\}$, on cherche quelle valeur de chaque attribut y se trouve (la valeur dont la position vaut $h$). Cela reconstitue le tableau classique de la solution.

In [4]:
use std::collections::HashMap;
use z3::ast::{Ast, Bool, Int};
use z3::Solver;

// Cellule autonome : sous evcxr les fonctions ne sont pas persistees d'une
// cellule a l'autre, le solveur est donc reconstruit ici.
// `skip` : numero (0..14) d'un indice a omettre — sert au bonus « sans un indice ».
fn build_solveur(skip: Option<usize>) -> (Solver, HashMap<String, Int>) {
    let mut s = Solver::new();
    // Les 5 groupes d'attributes, chacun avec ses 5 valeurs.
    let groups: Vec<Vec<&str>> = vec![
        vec!["Brit", "Swede", "Dane", "Norwegian", "German"],
        vec!["Red", "Green", "White", "Yellow", "Blue"],
        vec!["Tea", "Coffee", "Milk", "Water", "Beer"],
        vec!["PallMall", "Dunhill", "Blend", "BlueMaster", "Prince"],
        vec!["Dog", "Bird", "Cat", "Horse", "Fish"],
    ];
    // 25 variables de position (0..4) : une par valeur d'attribut.
    let mut v: HashMap<String, Int> = HashMap::new();
    for grp in &groups {
        for n in grp {
            v.insert(n.to_string(), Int::new_const(n.to_string().as_str()));
        }
    }
    // (1) Domaine 0..4 pour les 25 variables.
    for var in v.values() {
        s.assert(&var.ge(&Int::from_i64(0)));
        s.assert(&var.le(&Int::from_i64(4)));
    }
    // (2) Permutation : les 5 valeurs d'un meme attribut occupent des maisons distinctes.
    for grp in &groups {
        let vars: Vec<Int> = grp.iter().map(|n| v[*n].clone()).collect();
        s.assert(&Int::distinct(&vars));
    }
    // (3) Les 15 indices, dans l'ordre de l'enonce.
    let mut indices: Vec<(&str, Bool)> = Vec::new();
    indices.push(("1", v["Brit"]._eq(&v["Red"])));                        // le Britannique vit dans la rouge
    indices.push(("2", v["Swede"]._eq(&v["Dog"])));                       // le Suedois eleve des chiens
    indices.push(("3", v["Dane"]._eq(&v["Tea"])));                        // le Danois boit le the
    indices.push(("4", v["Green"]._eq(&Int::sub(&[&v["White"], &Int::from_i64(1)])))); // la verte est juste a gauche de la blanche
    indices.push(("5", v["Green"]._eq(&v["Coffee"])));                   // la verte boit le cafe
    indices.push(("6", v["PallMall"]._eq(&v["Bird"])));                  // Pall Mall eleve les oiseaux
    indices.push(("7", v["Yellow"]._eq(&v["Dunhill"])));                 // la jaune fume du Dunhill
    indices.push(("8", v["Milk"]._eq(&Int::from_i64(2))));               // la maison du milieu boit le lait
    indices.push(("9", v["Norwegian"]._eq(&Int::from_i64(0))));          // le Norvegien dans la 1ere maison
    // 10. Blend voisin du Chat
    let d1 = Int::sub(&[&v["Blend"], &v["Cat"]]);
    let d2 = Int::sub(&[&v["Cat"], &v["Blend"]]);
    indices.push(("10", Bool::or(&[&d1._eq(&Int::from_i64(1)), &d2._eq(&Int::from_i64(1))])));
    // 11. Cheval voisin du Dunhill
    let e1 = Int::sub(&[&v["Horse"], &v["Dunhill"]]);
    let e2 = Int::sub(&[&v["Dunhill"], &v["Horse"]]);
    indices.push(("11", Bool::or(&[&e1._eq(&Int::from_i64(1)), &e2._eq(&Int::from_i64(1))])));
    indices.push(("12", v["BlueMaster"]._eq(&v["Beer"])));                // BlueMaster boit de la biere
    indices.push(("13", v["German"]._eq(&v["Prince"])));                  // l'Allemand fume le Prince
    // 14. Norvegien voisin de la bleue
    let f1 = Int::sub(&[&v["Norwegian"], &v["Blue"]]);
    let f2 = Int::sub(&[&v["Blue"], &v["Norwegian"]]);
    indices.push(("14", Bool::or(&[&f1._eq(&Int::from_i64(1)), &f2._eq(&Int::from_i64(1))])));
    // 15. Blend voisin de l'Eau
    let g1 = Int::sub(&[&v["Blend"], &v["Water"]]);
    let g2 = Int::sub(&[&v["Water"], &v["Blend"]]);
    indices.push(("15", Bool::or(&[&g1._eq(&Int::from_i64(1)), &g2._eq(&Int::from_i64(1))])));

    for (i, (_numero, condition)) in indices.iter().enumerate() {
        if Some(i) == skip {
            continue;
        }
        s.assert(condition);
    }
    (s, v)
}

// Cas particulier : l'enigme complete, ses 15 indices.
fn build_einstein_solver() -> (Solver, HashMap<String, Int>) {
    build_solveur(None)
}

// Labels francais pour l'affichage (associes au nom de variable).
fn labels_fr() -> Vec<(&'static str, Vec<(&'static str, &'static str)>)> {
    vec![
        ("Nationalite", vec![("Britannique", "Brit"), ("Suedois", "Swede"), ("Danois", "Dane"), ("Norvegien", "Norwegian"), ("Allemand", "German")]),
        ("Couleur", vec![("Rouge", "Red"), ("Verte", "Green"), ("Blanche", "White"), ("Jaune", "Yellow"), ("Bleue", "Blue")]),
        ("Boisson", vec![("The", "Tea"), ("Cafe", "Coffee"), ("Lait", "Milk"), ("Eau", "Water"), ("Biere", "Beer")]),
        ("Cigarette", vec![("Pall Mall", "PallMall"), ("Dunhill", "Dunhill"), ("Blend", "Blend"), ("BlueMaster", "BlueMaster"), ("Prince", "Prince")]),
        ("Animal", vec![("Chien", "Dog"), ("Oiseau", "Bird"), ("Chat", "Cat"), ("Cheval", "Horse"), ("Poisson", "Fish")]),
    ]
}

// Retourne le label francais de la valeur du groupe presente a la maison h.
fn val_at(labels: &Vec<(&'static str, Vec<(&'static str, &'static str)>)>, pos: &HashMap<String, i64>, group: &str, h: i64) -> String {
    if let Some((_, valeurs)) = labels.iter().find(|(g, _)| *g == group) {
        for (fr, key) in valeurs {
            if pos[*key] == h {
                return fr.to_string();
            }
        }
    }
    "?".to_string()
}

// Resout l'enigme et affiche le tableau des 5 maisons.
fn solve_and_display(labels: &Vec<(&'static str, Vec<(&'static str, &'static str)>)>) -> HashMap<String, i64> {
    let (mut s, v) = build_einstein_solver();
    match s.check() {
        z3::SatResult::Sat => (),
        _ => {
            println!("UNSAT (inattendu pour cette enigme)");
            return HashMap::new();
        }
    }
    let m = match s.get_model() {
        Some(m) => m,
        None => {
            println!("Modele absent (inattendu)");
            return HashMap::new();
        }
    };
    // nom de variable -> position (0..4)
    let mut pos: HashMap<String, i64> = HashMap::new();
    for (nom, var) in v.iter() {
        let valeur = match m.eval(var, true).and_then(|i| i.as_i64()) {
            Some(x) => x,
            None => continue,
        };
        pos.insert(nom.clone(), valeur);
    }

    let cols = ["Maison", "Nationalite", "Couleur", "Boisson", "Cigarette", "Animal"];
    let widths = [7usize, 13, 9, 9, 13, 9];
    let header: Vec<String> = cols.iter().zip(widths.iter()).map(|(c, w)| format!("{:<width$}", c, width = w)).collect();
    println!("{}", header.join("  "));
    println!("  {}", "-".repeat(widths.iter().sum::<usize>() + 2 * (cols.len() - 1)));
    for h in 0..5 {
        let mut row = vec![format!("{:<width$}", h + 1, width = widths[0])];
        for (g, w) in ["Nationalite", "Couleur", "Boisson", "Cigarette", "Animal"].iter().zip(widths[1..].iter()) {
            row.push(format!("{:<width$}", val_at(labels, &pos, g, h), width = w));
        }
        println!("{}", row.join("  "));
    }
    println!();
    println!(">>> Reponse : c'est le {} qui possede le Poisson (maison {}).", val_at(labels, &pos, "Nationalite", pos["Fish"]), pos["Fish"] + 1);
    pos
}

let solution = solve_and_display(&labels_fr());

Maison   Nationalite    Couleur    Boisson    Cigarette      Animal   


  ----------------------------------------------------------------------


1        Norvegien      Jaune      Eau        Dunhill        Chat     


2        Danois         Bleue      The        Blend          Cheval   


3        Britannique    Rouge      Lait       Pall Mall      Oiseau   


4        Allemand       Verte      Cafe       Prince         Poisson  


5        Suedois        Blanche    Biere      BlueMaster     Chien    


>>> Reponse : c'est le Allemand qui possede le Poisson (maison 4).


## Interpretation

Z3 trouve l'unique solution en quelques millisecondes. La resolution humaine procederait par deductions progressives - on place le Norvegien en 1 (`Norwegian == 0`), le lait en 3 (`Milk == 2`), puis on propage les egalites et adjacences - mais c'est précisément ce travail de **propagation de contraintes** que le solveur automatise, sans erreur ni oubli.

La reponse a la question « qui possede le poisson ? » tombe directement du modèle extrait. Notons qu'**aucun indice ne parle du poisson** : c'est par elimination (les 4 autres animaux sont localises par les indices) que sa position se deduit.

## Vérification du témoin : relire les indices

Un modèle extrait n'est pas un résultat à prendre sur parole — vérifions que le tableau ci-dessus satisfait bien quelques indices choisis dans l'énoncé original :

- **Le Norvégien vit à côté de la maison bleue** : Norvégien en maison 1, Bleue en maison 2 — voisins.
- **Le fumeur de Blend a un voisin qui garde le chat** : Blend en maison 2, Chat en maison 1 — voisins.
- **La maison verte est immédiatement à gauche de la blanche** : Verte en 4, Blanche en 5.
- **La maison du centre boit du lait** : Lait en maison 3.

Ce geste de re-lecture croisée est l'équivalent d'un test unitaire sur le solveur : il valide l'**encodage** (une contrainte mal traduite produirait un témoin qui échoue à la vérification), pas seulement la satisfiabilité.

## Approche naive vs solveur : mesurer l'ecart



Plutot que d'affirmer que le solveur est plus rapide, **mesurons** l'ecart. L'espace de recherche brut est $(5!)^5$ : chaque attribut est une permutation des 5 maisons, independamment. On chronometre aussi la resolution Z3, puis on **prouve l'unicite** de la solution en ajoutant la negation du temoin trouve et en re-resolvant (on attend `unsat`).

In [5]:
use std::collections::HashMap;
use z3::ast::{Ast, Bool, Int};
use z3::Solver;

// Cellule autonome : sous evcxr les fonctions ne sont pas persistees d'une
// cellule a l'autre, le solveur est donc reconstruit ici.
// `skip` : numero (0..14) d'un indice a omettre — sert au bonus « sans un indice ».
fn build_solveur(skip: Option<usize>) -> (Solver, HashMap<String, Int>) {
    let mut s = Solver::new();
    // Les 5 groupes d'attributes, chacun avec ses 5 valeurs.
    let groups: Vec<Vec<&str>> = vec![
        vec!["Brit", "Swede", "Dane", "Norwegian", "German"],
        vec!["Red", "Green", "White", "Yellow", "Blue"],
        vec!["Tea", "Coffee", "Milk", "Water", "Beer"],
        vec!["PallMall", "Dunhill", "Blend", "BlueMaster", "Prince"],
        vec!["Dog", "Bird", "Cat", "Horse", "Fish"],
    ];
    // 25 variables de position (0..4) : une par valeur d'attribut.
    let mut v: HashMap<String, Int> = HashMap::new();
    for grp in &groups {
        for n in grp {
            v.insert(n.to_string(), Int::new_const(n.to_string().as_str()));
        }
    }
    // (1) Domaine 0..4 pour les 25 variables.
    for var in v.values() {
        s.assert(&var.ge(&Int::from_i64(0)));
        s.assert(&var.le(&Int::from_i64(4)));
    }
    // (2) Permutation : les 5 valeurs d'un meme attribut occupent des maisons distinctes.
    for grp in &groups {
        let vars: Vec<Int> = grp.iter().map(|n| v[*n].clone()).collect();
        s.assert(&Int::distinct(&vars));
    }
    // (3) Les 15 indices, dans l'ordre de l'enonce.
    let mut indices: Vec<(&str, Bool)> = Vec::new();
    indices.push(("1", v["Brit"]._eq(&v["Red"])));                        // le Britannique vit dans la rouge
    indices.push(("2", v["Swede"]._eq(&v["Dog"])));                       // le Suedois eleve des chiens
    indices.push(("3", v["Dane"]._eq(&v["Tea"])));                        // le Danois boit le the
    indices.push(("4", v["Green"]._eq(&Int::sub(&[&v["White"], &Int::from_i64(1)])))); // la verte est juste a gauche de la blanche
    indices.push(("5", v["Green"]._eq(&v["Coffee"])));                   // la verte boit le cafe
    indices.push(("6", v["PallMall"]._eq(&v["Bird"])));                  // Pall Mall eleve les oiseaux
    indices.push(("7", v["Yellow"]._eq(&v["Dunhill"])));                 // la jaune fume du Dunhill
    indices.push(("8", v["Milk"]._eq(&Int::from_i64(2))));               // la maison du milieu boit le lait
    indices.push(("9", v["Norwegian"]._eq(&Int::from_i64(0))));          // le Norvegien dans la 1ere maison
    // 10. Blend voisin du Chat
    let d1 = Int::sub(&[&v["Blend"], &v["Cat"]]);
    let d2 = Int::sub(&[&v["Cat"], &v["Blend"]]);
    indices.push(("10", Bool::or(&[&d1._eq(&Int::from_i64(1)), &d2._eq(&Int::from_i64(1))])));
    // 11. Cheval voisin du Dunhill
    let e1 = Int::sub(&[&v["Horse"], &v["Dunhill"]]);
    let e2 = Int::sub(&[&v["Dunhill"], &v["Horse"]]);
    indices.push(("11", Bool::or(&[&e1._eq(&Int::from_i64(1)), &e2._eq(&Int::from_i64(1))])));
    indices.push(("12", v["BlueMaster"]._eq(&v["Beer"])));                // BlueMaster boit de la biere
    indices.push(("13", v["German"]._eq(&v["Prince"])));                  // l'Allemand fume le Prince
    // 14. Norvegien voisin de la bleue
    let f1 = Int::sub(&[&v["Norwegian"], &v["Blue"]]);
    let f2 = Int::sub(&[&v["Blue"], &v["Norwegian"]]);
    indices.push(("14", Bool::or(&[&f1._eq(&Int::from_i64(1)), &f2._eq(&Int::from_i64(1))])));
    // 15. Blend voisin de l'Eau
    let g1 = Int::sub(&[&v["Blend"], &v["Water"]]);
    let g2 = Int::sub(&[&v["Water"], &v["Blend"]]);
    indices.push(("15", Bool::or(&[&g1._eq(&Int::from_i64(1)), &g2._eq(&Int::from_i64(1))])));

    for (i, (_numero, condition)) in indices.iter().enumerate() {
        if Some(i) == skip {
            continue;
        }
        s.assert(condition);
    }
    (s, v)
}

// Cas particulier : l'enigme complete, ses 15 indices.
fn build_einstein_solver() -> (Solver, HashMap<String, Int>) {
    build_solveur(None)
}

use std::time::Instant;

// (1) Espace de recherche brut
let space: u128 = 120u128 * 120 * 120 * 120 * 120; // (5!)^5 = 24_883_200_000
let brute_seconds = space as f64 / 1e8; // a ~1e8 tests/s, sans compter le test des 15 indices

// (2) Chronometre la resolution Z3
let t0 = Instant::now();
let (mut s1, v1) = build_einstein_solver();
let sat1 = s1.check();
let m1 = s1.get_model();
let z3_ms = t0.elapsed().as_secs_f64() * 1000.0;

// (3) Unicite : on nie le temoin trouve (au moins une variable differe) -> on attend unsat
let (mut s2, v2) = build_einstein_solver();
let mut clauses: Vec<Bool> = Vec::new();
if let Some(m) = &m1 {
    for (nom, var) in v2.iter() {
        let valeur = m.eval(&v1[nom], true).and_then(|i| i.as_i64()).unwrap_or(0);
        clauses.push(var._eq(&Int::from_i64(valeur)).not());
    }
}
s2.assert(&Bool::or(&clauses));
let unique = s2.check() == z3::SatResult::Unsat;

// 24 883 200 000 avec les separateurs de milliers de l'espace de recherche.
fn avec_separateurs(n: u128) -> String {
    let brut = n.to_string();
    let mut out = String::new();
    for (i, c) in brut.chars().enumerate() {
        if i > 0 && (brut.len() - i) % 3 == 0 {
            out.push(',');
        }
        out.push(c);
    }
    out
}

println!("Espace de recherche brut       : (5!)^5 = {} combinaisons", avec_separateurs(space));
println!("Brute force estime (~1e8/s)    : {:.1} s (hors test des 15 indices)", brute_seconds);
println!("Z3 solve temps                 : {:.1} ms", z3_ms);
println!("Solution unique (UNSAT si nie) : {}", unique);
println!(" -> Z3 resout en millisecondes un espace de ~{:.0} milliards, et prouve l'unicite sans enumeration.", space as f64 / 1e9);
let _ = sat1;

Espace de recherche brut       : (5!)^5 = 24,883,200,000 combinaisons


Brute force estime (~1e8/s)    : 248.8 s (hors test des 15 indices)


Z3 solve temps                 : 9.6 ms


Solution unique (UNSAT si nie) : true


 -> Z3 resout en millisecondes un espace de ~25 milliards, et prouve l'unicite sans enumeration.


## Lecture du benchmark

Sur cette enigme, l'ecart est **frappant** : Z3 resout les 25 variables entrelacees en quelques millisecondes, alors que la brute force doit affronter **~25 milliards** de combinaisons (soit plusieurs minutes même a cadence elevee, et bien plus avec le test des 15 indices a chaque candidat).

La **propagation** fait tout le travail : `Norwegian == 0` et `Milk == 2` sont des ancres imposees par les indices 9 et 8, puis les egalites de position et les adjacences reduisent les domaines restants jusqu'a ce qu'il ne reste qu'une seule combinaison coherente. C'est exactement la mecanique qu'un humain deployerait a la main sur le tableau - mais le solveur la realise sans erreur et la **prouve complete** (unicite via `unsat` sur la negation).

## Exercices

Les trois exercices initiaux de ce carnet ont ete resolus par un rendu de TP : chacun devient un **exemple guide** (code conserve tel quel), suivi d'une lecture du resultat puis d'un **nouvel exercice** non resolu, de difficulte comparable et portant le meme concept Z3. Vous y reutiliserez l'encodage par position et la fonction `build_einstein_solver`.

Les trois exemples guides sont ecrits en Python dans le jumeau : ici, ils sont **portes en Rust** (meme logique, meme sortie affichee). Les enonces des trois nouveaux exercices sont transposes : une `lambda v: ...` Python devient une closure Rust `|v| ...` qui rend un `Bool` Z3.


### Exemple guide 1 -- Un 16e indice : le Chat boit de l'Eau

Resolu par @EmmaLehec (TP2 IA2-Min2, groupe G02, PR #27026), integre comme exemple guide.

**Objectif** : ajouter l'indice `Cat == Water` (le Chat et l'Eau sont dans la meme maison) et observer si l'enigme reste satisfaisable (`sat`) ou devient contradictoire (`unsat`).

**Rappel** : on reprend `build_einstein_solver()`, on ajoute `s.assert(&v["Cat"]._eq(&v["Water"]))` avant `check()`, et on compare avec la solution trouvee plus haut — le Chat et l'Eau y sont-ils dans la meme maison ?


In [6]:
use std::collections::HashMap;
use z3::ast::{Ast, Bool, Int};
use z3::Solver;

// Cellule autonome : sous evcxr les fonctions ne sont pas persistees d'une
// cellule a l'autre, le solveur est donc reconstruit ici.
// `skip` : numero (0..14) d'un indice a omettre — sert au bonus « sans un indice ».
fn build_solveur(skip: Option<usize>) -> (Solver, HashMap<String, Int>) {
    let mut s = Solver::new();
    // Les 5 groupes d'attributes, chacun avec ses 5 valeurs.
    let groups: Vec<Vec<&str>> = vec![
        vec!["Brit", "Swede", "Dane", "Norwegian", "German"],
        vec!["Red", "Green", "White", "Yellow", "Blue"],
        vec!["Tea", "Coffee", "Milk", "Water", "Beer"],
        vec!["PallMall", "Dunhill", "Blend", "BlueMaster", "Prince"],
        vec!["Dog", "Bird", "Cat", "Horse", "Fish"],
    ];
    // 25 variables de position (0..4) : une par valeur d'attribut.
    let mut v: HashMap<String, Int> = HashMap::new();
    for grp in &groups {
        for n in grp {
            v.insert(n.to_string(), Int::new_const(n.to_string().as_str()));
        }
    }
    // (1) Domaine 0..4 pour les 25 variables.
    for var in v.values() {
        s.assert(&var.ge(&Int::from_i64(0)));
        s.assert(&var.le(&Int::from_i64(4)));
    }
    // (2) Permutation : les 5 valeurs d'un meme attribut occupent des maisons distinctes.
    for grp in &groups {
        let vars: Vec<Int> = grp.iter().map(|n| v[*n].clone()).collect();
        s.assert(&Int::distinct(&vars));
    }
    // (3) Les 15 indices, dans l'ordre de l'enonce.
    let mut indices: Vec<(&str, Bool)> = Vec::new();
    indices.push(("1", v["Brit"]._eq(&v["Red"])));                        // le Britannique vit dans la rouge
    indices.push(("2", v["Swede"]._eq(&v["Dog"])));                       // le Suedois eleve des chiens
    indices.push(("3", v["Dane"]._eq(&v["Tea"])));                        // le Danois boit le the
    indices.push(("4", v["Green"]._eq(&Int::sub(&[&v["White"], &Int::from_i64(1)])))); // la verte est juste a gauche de la blanche
    indices.push(("5", v["Green"]._eq(&v["Coffee"])));                   // la verte boit le cafe
    indices.push(("6", v["PallMall"]._eq(&v["Bird"])));                  // Pall Mall eleve les oiseaux
    indices.push(("7", v["Yellow"]._eq(&v["Dunhill"])));                 // la jaune fume du Dunhill
    indices.push(("8", v["Milk"]._eq(&Int::from_i64(2))));               // la maison du milieu boit le lait
    indices.push(("9", v["Norwegian"]._eq(&Int::from_i64(0))));          // le Norvegien dans la 1ere maison
    // 10. Blend voisin du Chat
    let d1 = Int::sub(&[&v["Blend"], &v["Cat"]]);
    let d2 = Int::sub(&[&v["Cat"], &v["Blend"]]);
    indices.push(("10", Bool::or(&[&d1._eq(&Int::from_i64(1)), &d2._eq(&Int::from_i64(1))])));
    // 11. Cheval voisin du Dunhill
    let e1 = Int::sub(&[&v["Horse"], &v["Dunhill"]]);
    let e2 = Int::sub(&[&v["Dunhill"], &v["Horse"]]);
    indices.push(("11", Bool::or(&[&e1._eq(&Int::from_i64(1)), &e2._eq(&Int::from_i64(1))])));
    indices.push(("12", v["BlueMaster"]._eq(&v["Beer"])));                // BlueMaster boit de la biere
    indices.push(("13", v["German"]._eq(&v["Prince"])));                  // l'Allemand fume le Prince
    // 14. Norvegien voisin de la bleue
    let f1 = Int::sub(&[&v["Norwegian"], &v["Blue"]]);
    let f2 = Int::sub(&[&v["Blue"], &v["Norwegian"]]);
    indices.push(("14", Bool::or(&[&f1._eq(&Int::from_i64(1)), &f2._eq(&Int::from_i64(1))])));
    // 15. Blend voisin de l'Eau
    let g1 = Int::sub(&[&v["Blend"], &v["Water"]]);
    let g2 = Int::sub(&[&v["Water"], &v["Blend"]]);
    indices.push(("15", Bool::or(&[&g1._eq(&Int::from_i64(1)), &g2._eq(&Int::from_i64(1))])));

    for (i, (_numero, condition)) in indices.iter().enumerate() {
        if Some(i) == skip {
            continue;
        }
        s.assert(condition);
    }
    (s, v)
}

// Cas particulier : l'enigme complete, ses 15 indices.
fn build_einstein_solver() -> (Solver, HashMap<String, Int>) {
    build_solveur(None)
}

// Exemple guide 1 : ajout de l'indice Cat == Water et test de la satisfiabilite.
// Etape 1 : reprenez build_einstein_solver() -> (s, v).
// Etape 2 : s.assert(&v["Cat"]._eq(&v["Water"])).
// Etape 3 : s.check() -> sat ou unsat ?

fn tester_chat_eau() -> String {
    // Etape 1 : On recupere le solveur deja configure avec les 15 indices
    let (mut s, v) = build_einstein_solver();

    // Etape 2 : On ajoute le 16e indice (le Chat et l'Eau dans la meme maison)
    s.assert(&v["Cat"]._eq(&v["Water"]));

    // Etape 3 : On verifie si le probleme reste possible et on renvoie le resultat
    match s.check() {
        z3::SatResult::Sat => "sat".to_string(),
        z3::SatResult::Unsat => "unsat".to_string(),
        z3::SatResult::Unknown => "unknown".to_string(),
    }
}

// Pour tester ta fonction et afficher le resultat :
println!("Verdict avec Cat == Water : {}", tester_chat_eau());

Verdict avec Cat == Water : sat


### Lecture du résultat : un indice déjà vrai

Le verdict est `sat` : l'énigme reste satisfaisable. Ce n'est pas un hasard — dans l'unique solution affichée plus haut, le Chat et l'Eau sont **déjà** dans la même maison (maison 1, chez le Norvégien). Le 16e indice est donc déjà vrai : il ne crée aucune contradiction et n'apporte aucune information nouvelle. Un indice qui contredirait la solution (par exemple `Fish == Water`) rendrait au contraire l'énigme `unsat` — c'est exactement l'objet du nouvel exercice 1.

### Exercice 1 -- Predire le verdict d'un 17e indice

**Objectif** : generaliser l'exemple guide 1 en une fonction qui classe **n'importe quel** indice candidat, puis repondre a la question conceptuelle.

Ecrire `classer_indice(indice)` qui prend un indice candidat (une closure qui recoit la table des variables `v` et rend une expression booleenne Z3), reconstruit le solveur complet, ajoute l'indice et retourne `"sat"` ou `"unsat"`. Classer ensuite les quatre candidats :

1. `|v| v["Cat"]._eq(&v["Water"])` — le Chat et l'Eau dans la meme maison ;
2. `|v| v["Fish"]._eq(&v["Water"])` — le Poisson et l'Eau dans la meme maison ;
3. `|v| v["Horse"]._eq(&v["Blend"])` — le Cheval dans la maison du fumeur de Blend ;
4. `|v| v["Bird"]._eq(&v["Milk"])` — l'Oiseau dans la maison au lait.

**Question** : parmi les verdicts obtenus, peut-il exister un indice qui donne `sat` tout en **changeant** la solution du tableau ? Justifier a partir de l'unicite prouvee plus haut.

**Indices** : chaque appel doit repartir d'un solveur **frais** (`build_einstein_solver()`) — sinon les indices s'accumulent d'un appel a l'autre. Pour la question : que vaut l'ensemble des solutions avant l'ajout, et que lui fait un indice deja vrai ?


In [7]:
use std::collections::HashMap;
use z3::ast::{Bool, Int};

// Exercice 1 : classer un 17e indice candidat (sat / unsat).
// Etape 1 : une fonction classer_indice(indice) qui repart d'un solveur FRAIS a chaque appel.
// Etape 2 : classer les 4 indices candidats de l'enonce.
// Etape 3 : repondre a la question -- un indice sat peut-il changer la solution ?

// TODO etudiant : reconstruisez le solveur, ajoutez indice(v), retournez le verdict.
// L'indice est une closure qui prend la table des variables et rend une expression Z3.
fn classer_indice<F>(_indice: F) -> Option<&'static str>
where
    F: Fn(&HashMap<String, Int>) -> Bool,
{
    None // TODO etudiant
}

println!("Exercice a completer : classer les 4 indices candidats.");

Exercice a completer : classer les 4 indices candidats.


### Exemple guide 2 -- Enumerer toutes les solutions par blocage iteratif

Resolu par @EmmaLehec (TP2 IA2-Min2, groupe G02, PR #27026), integre comme exemple guide.

**Objectif** : enumerer **toutes** les solutions de l'enigme par **blocage iteratif** (on resout, on compte, on ajoute une contrainte interdisant le temoin trouve, on re-resout) jusqu'a `unsat`. Le compte attendu est 1 (solution unique), ce qui confirme le benchmark ci-dessus. Le rendu generalise la fonction en `compter_solutions(s, v, limite)` pour qu'elle s'applique a n'importe quel solveur — elle resservira dans l'exemple guide 3 et le nouvel exercice 2.


In [8]:
use std::collections::HashMap;
use z3::ast::{Ast, Bool, Int};
use z3::Solver;

// Cellule autonome : sous evcxr les fonctions ne sont pas persistees d'une
// cellule a l'autre, le solveur est donc reconstruit ici.
// `skip` : numero (0..14) d'un indice a omettre — sert au bonus « sans un indice ».
fn build_solveur(skip: Option<usize>) -> (Solver, HashMap<String, Int>) {
    let mut s = Solver::new();
    // Les 5 groupes d'attributes, chacun avec ses 5 valeurs.
    let groups: Vec<Vec<&str>> = vec![
        vec!["Brit", "Swede", "Dane", "Norwegian", "German"],
        vec!["Red", "Green", "White", "Yellow", "Blue"],
        vec!["Tea", "Coffee", "Milk", "Water", "Beer"],
        vec!["PallMall", "Dunhill", "Blend", "BlueMaster", "Prince"],
        vec!["Dog", "Bird", "Cat", "Horse", "Fish"],
    ];
    // 25 variables de position (0..4) : une par valeur d'attribut.
    let mut v: HashMap<String, Int> = HashMap::new();
    for grp in &groups {
        for n in grp {
            v.insert(n.to_string(), Int::new_const(n.to_string().as_str()));
        }
    }
    // (1) Domaine 0..4 pour les 25 variables.
    for var in v.values() {
        s.assert(&var.ge(&Int::from_i64(0)));
        s.assert(&var.le(&Int::from_i64(4)));
    }
    // (2) Permutation : les 5 valeurs d'un meme attribut occupent des maisons distinctes.
    for grp in &groups {
        let vars: Vec<Int> = grp.iter().map(|n| v[*n].clone()).collect();
        s.assert(&Int::distinct(&vars));
    }
    // (3) Les 15 indices, dans l'ordre de l'enonce.
    let mut indices: Vec<(&str, Bool)> = Vec::new();
    indices.push(("1", v["Brit"]._eq(&v["Red"])));                        // le Britannique vit dans la rouge
    indices.push(("2", v["Swede"]._eq(&v["Dog"])));                       // le Suedois eleve des chiens
    indices.push(("3", v["Dane"]._eq(&v["Tea"])));                        // le Danois boit le the
    indices.push(("4", v["Green"]._eq(&Int::sub(&[&v["White"], &Int::from_i64(1)])))); // la verte est juste a gauche de la blanche
    indices.push(("5", v["Green"]._eq(&v["Coffee"])));                   // la verte boit le cafe
    indices.push(("6", v["PallMall"]._eq(&v["Bird"])));                  // Pall Mall eleve les oiseaux
    indices.push(("7", v["Yellow"]._eq(&v["Dunhill"])));                 // la jaune fume du Dunhill
    indices.push(("8", v["Milk"]._eq(&Int::from_i64(2))));               // la maison du milieu boit le lait
    indices.push(("9", v["Norwegian"]._eq(&Int::from_i64(0))));          // le Norvegien dans la 1ere maison
    // 10. Blend voisin du Chat
    let d1 = Int::sub(&[&v["Blend"], &v["Cat"]]);
    let d2 = Int::sub(&[&v["Cat"], &v["Blend"]]);
    indices.push(("10", Bool::or(&[&d1._eq(&Int::from_i64(1)), &d2._eq(&Int::from_i64(1))])));
    // 11. Cheval voisin du Dunhill
    let e1 = Int::sub(&[&v["Horse"], &v["Dunhill"]]);
    let e2 = Int::sub(&[&v["Dunhill"], &v["Horse"]]);
    indices.push(("11", Bool::or(&[&e1._eq(&Int::from_i64(1)), &e2._eq(&Int::from_i64(1))])));
    indices.push(("12", v["BlueMaster"]._eq(&v["Beer"])));                // BlueMaster boit de la biere
    indices.push(("13", v["German"]._eq(&v["Prince"])));                  // l'Allemand fume le Prince
    // 14. Norvegien voisin de la bleue
    let f1 = Int::sub(&[&v["Norwegian"], &v["Blue"]]);
    let f2 = Int::sub(&[&v["Blue"], &v["Norwegian"]]);
    indices.push(("14", Bool::or(&[&f1._eq(&Int::from_i64(1)), &f2._eq(&Int::from_i64(1))])));
    // 15. Blend voisin de l'Eau
    let g1 = Int::sub(&[&v["Blend"], &v["Water"]]);
    let g2 = Int::sub(&[&v["Water"], &v["Blend"]]);
    indices.push(("15", Bool::or(&[&g1._eq(&Int::from_i64(1)), &g2._eq(&Int::from_i64(1))])));

    for (i, (_numero, condition)) in indices.iter().enumerate() {
        if Some(i) == skip {
            continue;
        }
        s.assert(condition);
    }
    (s, v)
}

// Cas particulier : l'enigme complete, ses 15 indices.
fn build_einstein_solver() -> (Solver, HashMap<String, Int>) {
    build_solveur(None)
}
// Cellule autonome : le blocage iteratif est reecrit ici.
fn compter_solutions(s: &mut Solver, v: &HashMap<String, Int>, limite: usize) -> usize {
    let mut compte = 0usize;
    while compte < limite && s.check() == z3::SatResult::Sat {
        compte += 1;
        let m = match s.get_model() {
            Some(m) => m,
            None => break,
        };
        // On interdit le temoin trouve : au moins une variable doit changer.
        let mut clauses: Vec<Bool> = Vec::new();
        for var in v.values() {
            let valeur = m.eval(var, true).and_then(|i| i.as_i64()).unwrap_or(0);
            clauses.push(var._eq(&Int::from_i64(valeur)).not());
        }
        s.assert(&Bool::or(&clauses));
    }
    compte
}

// Exemple guide 2 : enumerer toutes les solutions par blocage iteratif.
// Etape 3 : afficher le compte (attendu : 1).
let (mut s, v) = build_einstein_solver();
let nb = compter_solutions(&mut s, &v, 1000);
println!("Nombre de solutions de l'enigme d'Einstein : {} (attendu : 1)", nb);

Nombre de solutions de l'enigme d'Einstein : 1 (attendu : 1)


In [9]:
use std::collections::HashMap;
use z3::ast::{Ast, Bool, Int};
use z3::Solver;

// Cellule autonome : sous evcxr les fonctions ne sont pas persistees d'une
// cellule a l'autre, le solveur est donc reconstruit ici.
// `skip` : numero (0..14) d'un indice a omettre — sert au bonus « sans un indice ».
fn build_solveur(skip: Option<usize>) -> (Solver, HashMap<String, Int>) {
    let mut s = Solver::new();
    // Les 5 groupes d'attributes, chacun avec ses 5 valeurs.
    let groups: Vec<Vec<&str>> = vec![
        vec!["Brit", "Swede", "Dane", "Norwegian", "German"],
        vec!["Red", "Green", "White", "Yellow", "Blue"],
        vec!["Tea", "Coffee", "Milk", "Water", "Beer"],
        vec!["PallMall", "Dunhill", "Blend", "BlueMaster", "Prince"],
        vec!["Dog", "Bird", "Cat", "Horse", "Fish"],
    ];
    // 25 variables de position (0..4) : une par valeur d'attribut.
    let mut v: HashMap<String, Int> = HashMap::new();
    for grp in &groups {
        for n in grp {
            v.insert(n.to_string(), Int::new_const(n.to_string().as_str()));
        }
    }
    // (1) Domaine 0..4 pour les 25 variables.
    for var in v.values() {
        s.assert(&var.ge(&Int::from_i64(0)));
        s.assert(&var.le(&Int::from_i64(4)));
    }
    // (2) Permutation : les 5 valeurs d'un meme attribut occupent des maisons distinctes.
    for grp in &groups {
        let vars: Vec<Int> = grp.iter().map(|n| v[*n].clone()).collect();
        s.assert(&Int::distinct(&vars));
    }
    // (3) Les 15 indices, dans l'ordre de l'enonce.
    let mut indices: Vec<(&str, Bool)> = Vec::new();
    indices.push(("1", v["Brit"]._eq(&v["Red"])));                        // le Britannique vit dans la rouge
    indices.push(("2", v["Swede"]._eq(&v["Dog"])));                       // le Suedois eleve des chiens
    indices.push(("3", v["Dane"]._eq(&v["Tea"])));                        // le Danois boit le the
    indices.push(("4", v["Green"]._eq(&Int::sub(&[&v["White"], &Int::from_i64(1)])))); // la verte est juste a gauche de la blanche
    indices.push(("5", v["Green"]._eq(&v["Coffee"])));                   // la verte boit le cafe
    indices.push(("6", v["PallMall"]._eq(&v["Bird"])));                  // Pall Mall eleve les oiseaux
    indices.push(("7", v["Yellow"]._eq(&v["Dunhill"])));                 // la jaune fume du Dunhill
    indices.push(("8", v["Milk"]._eq(&Int::from_i64(2))));               // la maison du milieu boit le lait
    indices.push(("9", v["Norwegian"]._eq(&Int::from_i64(0))));          // le Norvegien dans la 1ere maison
    // 10. Blend voisin du Chat
    let d1 = Int::sub(&[&v["Blend"], &v["Cat"]]);
    let d2 = Int::sub(&[&v["Cat"], &v["Blend"]]);
    indices.push(("10", Bool::or(&[&d1._eq(&Int::from_i64(1)), &d2._eq(&Int::from_i64(1))])));
    // 11. Cheval voisin du Dunhill
    let e1 = Int::sub(&[&v["Horse"], &v["Dunhill"]]);
    let e2 = Int::sub(&[&v["Dunhill"], &v["Horse"]]);
    indices.push(("11", Bool::or(&[&e1._eq(&Int::from_i64(1)), &e2._eq(&Int::from_i64(1))])));
    indices.push(("12", v["BlueMaster"]._eq(&v["Beer"])));                // BlueMaster boit de la biere
    indices.push(("13", v["German"]._eq(&v["Prince"])));                  // l'Allemand fume le Prince
    // 14. Norvegien voisin de la bleue
    let f1 = Int::sub(&[&v["Norwegian"], &v["Blue"]]);
    let f2 = Int::sub(&[&v["Blue"], &v["Norwegian"]]);
    indices.push(("14", Bool::or(&[&f1._eq(&Int::from_i64(1)), &f2._eq(&Int::from_i64(1))])));
    // 15. Blend voisin de l'Eau
    let g1 = Int::sub(&[&v["Blend"], &v["Water"]]);
    let g2 = Int::sub(&[&v["Water"], &v["Blend"]]);
    indices.push(("15", Bool::or(&[&g1._eq(&Int::from_i64(1)), &g2._eq(&Int::from_i64(1))])));

    for (i, (_numero, condition)) in indices.iter().enumerate() {
        if Some(i) == skip {
            continue;
        }
        s.assert(condition);
    }
    (s, v)
}

// Cas particulier : l'enigme complete, ses 15 indices.
fn build_einstein_solver() -> (Solver, HashMap<String, Int>) {
    build_solveur(None)
}
// Cellule autonome : le blocage iteratif est reecrit ici.
fn compter_solutions(s: &mut Solver, v: &HashMap<String, Int>, limite: usize) -> usize {
    let mut compte = 0usize;
    while compte < limite && s.check() == z3::SatResult::Sat {
        compte += 1;
        let m = match s.get_model() {
            Some(m) => m,
            None => break,
        };
        // On interdit le temoin trouve : au moins une variable doit changer.
        let mut clauses: Vec<Bool> = Vec::new();
        for var in v.values() {
            let valeur = m.eval(var, true).and_then(|i| i.as_i64()).unwrap_or(0);
            clauses.push(var._eq(&Int::from_i64(valeur)).not());
        }
        s.assert(&Bool::or(&clauses));
    }
    compte
}

// Bonus exemple guide 2 : la boucle compte-t-elle vraiment ? On retire un indice et on recompte.
// Le solveur est reconstruit sans l'indice retire (le numero designe sa place
// dans la liste des 15 indices de l'enonce).
let tests: Vec<(&str, usize)> = vec![
    ("9  (Norvegien en maison 1)", 8),
    ("15 (Blend voisin de l'Eau)", 14),
];
let mut texte = String::new();
for (nom, numero) in tests {
    let (mut s2, v2) = build_solveur(Some(numero));
    texte.push_str(&format!("Sans l'indice {} : {} solution(s)\n", nom, compter_solutions(&mut s2, &v2, 1000)));
}
print!("{}", texte);

Sans l'indice 9  (Norvegien en maison 1) : 17 solution(s)


Sans l'indice 15 (Blend voisin de l'Eau) : 1 solution(s)


### Lecture du résultat : la boucle compte vraiment, et un indice est redondant

Le blocage itératif (trouver une solution, l'interdire par un `Or` sur les 25 variables, recommencer jusqu'à `unsat`) donne **1 solution** : l'énigme d'Einstein est bien unique, ce qui confirme le benchmark. Pour vérifier que la boucle compte réellement, on retire un indice et on recompte : sans l'indice 9 (Norvégien en maison 1), on trouve **17 solutions**. Sans l'indice 15 (Blend voisin de l'Eau), il en reste **1** : cet indice est **redondant**, les 14 autres suffisent à déterminer la solution. Le nouvel exercice 2 pousse cette question un cran plus loin.

### Exercice 2 -- Un sous-ensemble minimal d'indices

**Objectif** : l'indice 15 est redondant, mais il n'est probablement pas le seul. Trouver par **elimination gloutonne** un sous-ensemble des 15 indices qui garde **exactement 1 solution** : partir de la liste complete, et pour chaque indice dans l'ordre, essayer de le retirer **definitivement** — si le reste a toujours exactement 1 solution, il reste retire, sinon il est remis.

**Indices** : listez d'abord les 15 indices sous forme de closures `|v| <expression Z3>` (les deux du bonus donnent le patron) ; reconstruisez a chaque essai un solveur frais ne contenant que les indices retenus, et comptez avec `compter_solutions(s, v, limite)`. Affichez le nombre d'indices conserves et leur liste. Attention : un indice isolement redondant peut devenir necessaire une fois d'autres indices retires — d'ou le test a chaque etape.


In [10]:
// Exercice 2 : trouver un sous-ensemble minimal d'indices.
// TODO etudiant : liste des 15 indices, puis boucle d'elimination gloutonne.
fn indices_minimaux() -> Option<Vec<&'static str>> {
    None // TODO etudiant
}

println!("Exercice a completer : élimination gloutonne des indices redondants.");

Exercice a completer : élimination gloutonne des indices redondants.


### Exemple guide 3 -- Mini-Zebra a 4 maisons

Resolu par @EmmaLehec (TP2 IA2-Min2, groupe G02, PR #27026), integre comme exemple guide.

**Objectif** : modeliser une **mini-enigme** a 4 maisons et 3 attributs (4 nationalites, 4 couleurs, 4 animaux) avec 8 indices, et verifier que l'encodage par position se generalise — le modele est parametre par les donnees, seule l'instance change.


In [11]:
use std::collections::HashMap;
use z3::ast::{Ast, Bool, Int};
use z3::Solver;

// Cellule autonome : le mini-Zebra a son propre solveur et son propre
// compteur de solutions (les fonctions ne sont pas persistees sous evcxr).

// Un mini-Zebra a 4 maisons / 3 attributs.
fn resoudre_mini_zebra() -> (Solver, HashMap<String, Int>) {
    let mut s = Solver::new();
    // 1. Definition des groupes
    let groups_4: Vec<Vec<&str>> = vec![
        vec!["Francais", "Anglais", "Espagnol", "Italien"],
        vec!["Rouge", "Bleu", "Vert", "Jaune"],
        vec!["Chien", "Chat", "Poisson", "Oiseau"],
    ];

    // Creation des variables
    let mut v: HashMap<String, Int> = HashMap::new();
    for grp in &groups_4 {
        for nom in grp {
            v.insert(nom.to_string(), Int::new_const(nom.to_string().as_str()));
        }
    }

    // 2. Domaine : chaque variable doit etre entre 0 et 3
    for var in v.values() {
        s.assert(&var.ge(&Int::from_i64(0)));
        s.assert(&var.le(&Int::from_i64(3)));
    }

    // Les 4 valeurs de chaque attribut doivent etre dans des maisons differentes
    for grp in &groups_4 {
        let vars: Vec<Int> = grp.iter().map(|nom| v[*nom].clone()).collect();
        s.assert(&Int::distinct(&vars));
    }

    // 3. Les 8 indices

    // 1. Le Francais habite dans la maison rouge
    s.assert(&v["Francais"]._eq(&v["Rouge"]));

    // 2. l'Anglais possede le chien
    s.assert(&v["Anglais"]._eq(&v["Chien"]));

    // 3. l'Espagnol habite dans la maison verte
    s.assert(&v["Espagnol"]._eq(&v["Vert"]));

    // 4. Le Francais possede le chat
    s.assert(&v["Francais"]._eq(&v["Chat"]));

    // 5. L'Espagnol habite dans la premiere maison
    s.assert(&v["Espagnol"]._eq(&Int::from_i64(0)));

    // 6. Le chien est a cote de la maison jaune
    let c6a = Int::sub(&[&v["Chien"], &v["Jaune"]]);
    let c6b = Int::sub(&[&v["Jaune"], &v["Chien"]]);
    s.assert(&Bool::or(&[&c6a._eq(&Int::from_i64(1)), &c6b._eq(&Int::from_i64(1))]));

    // 7. Le poisson est dans la maison verte
    s.assert(&v["Poisson"]._eq(&v["Vert"]));

    // 8. La maison bleue est a cote de la maison verte
    let c8a = Int::sub(&[&v["Bleu"], &v["Vert"]]);
    let c8b = Int::sub(&[&v["Vert"], &v["Bleu"]]);
    s.assert(&Bool::or(&[&c8a._eq(&Int::from_i64(1)), &c8b._eq(&Int::from_i64(1))]));

    (s, v)
}

// Compteur de solutions (cellule autonome : reecrit ici).
fn compter_solutions(s: &mut Solver, v: &HashMap<String, Int>, limite: usize) -> usize {
    let mut compte = 0usize;
    while compte < limite && s.check() == z3::SatResult::Sat {
        compte += 1;
        let m = match s.get_model() {
            Some(m) => m,
            None => break,
        };
        let mut clauses: Vec<Bool> = Vec::new();
        for var in v.values() {
            let valeur = m.eval(var, true).and_then(|i| i.as_i64()).unwrap_or(0);
            clauses.push(var._eq(&Int::from_i64(valeur)).not());
        }
        s.assert(&Bool::or(&clauses));
    }
    compte
}

let groupes: Vec<(&str, Vec<&str>)> = vec![
    ("Nationalité", vec!["Francais", "Anglais", "Espagnol", "Italien"]),
    ("Couleur", vec!["Rouge", "Bleu", "Vert", "Jaune"]),
    ("Animal", vec!["Chien", "Chat", "Poisson", "Oiseau"]),
];

// Test du solveur
let (mut s, v) = resoudre_mini_zebra();

match s.check() {
    z3::SatResult::Sat => {
        println!("Solution trouvée !");
        if let Some(m) = s.get_model() {
            let mut pos: HashMap<String, i64> = HashMap::new();
            for (nom, var) in v.iter() {
                if let Some(valeur) = m.eval(var, true).and_then(|i| i.as_i64()) {
                    pos.insert(nom.clone(), valeur);
                }
            }
            // Affichage des maisons
            for maison in 0..4 {
                println!("\nMaison {} :", maison + 1);
                for (groupe, valeurs) in &groupes {
                    for valeur in valeurs {
                        if pos[*valeur] == maison as i64 {
                            println!("  {} : {}", groupe, valeur);
                        }
                    }
                }
            }
        } else {
            println!("Modele absent (inattendu)");
        }
    }
    _ => println!("Aucune solution."),
}

// Etape 3 : solution unique ? On reutilise compter_solutions (exemple guide 2).
let (mut s_u, v_u) = resoudre_mini_zebra();
println!("\nNombre de solutions : {} (attendu : 1)", compter_solutions(&mut s_u, &v_u, 1000));

Solution trouvée !


Maison 1 :


  Nationalité : Espagnol


  Couleur : Vert


  Animal : Poisson


Maison 2 :


... (8 outputs left out)

  Nationalité : Francais


  Couleur : Rouge


  Animal : Chat


Nombre de solutions : 1 (attendu : 1)


### Lecture du résultat : l'encodage se généralise tel quel

Le même encodage par position passe directement à 4 maisons : domaine `0..3`, 3 groupes `Ast::distinct` et 8 indices (égalités de position et adjacences). Trouver une solution (`sat`) ne suffit pas pour une énigme : il faut aussi qu'elle soit **unique**. En réutilisant `compter_solutions` (exemple guidé 2), on vérifie que les 8 indices déterminent **une seule solution** : l'Espagnol, dans la maison verte, possède le poisson.

### Exercice 3 -- Cinq amies, cinq boissons, cinq loisirs

**Objectif** : modeliser une enigme a **5 maisons** sur un autre theme, avec deux types de relations **nouveaux** par rapport au mini-Zebra : « a gauche quelque part » (`a.lt(&b)`) et « immediatement a gauche de » (`a._eq(&Int::sub(&[&b, &Int::from_i64(1)]))`). Cinq amies (Alice, Bruno, Chloe, David, Emilie), cinq boissons (Cafe, The, Jus, Lait, Eau) et cinq loisirs (Echecs, Lecture, Course, Peinture, Musique) — une valeur par maison, une maison par valeur.

Les 11 indices :

1. Alice habite la premiere maison.
2. Emilie habite la derniere maison.
3. Qui boit du lait fait de la peinture.
4. Bruno boit du the.
5. Le joueur d'echecs habite quelque part a gauche du buveur de cafe.
6. David boit le jus.
7. Le coureur habite immediatement a gauche de Chloe.
8. La lectrice boit de l'eau.
9. Chloe fait de la musique.
10. Le buveur de cafe habite immediatement a droite de Bruno.
11. La peinture se pratique dans une maison a gauche de celle de Chloe.

**Indices** : reprenez la structure de `resoudre_mini_zebra` (domaine `0..4`, 3 groupes `Ast::distinct`) ; verifiez `sat` **puis** l'unicite avec `compter_solutions(s, v, limite)` — attendu : exactement 1 solution.


In [12]:
use std::collections::HashMap;
use z3::ast::Int;
use z3::Solver;

// Exercice 3 : cinq amies, cinq boissons, cinq loisirs (5 maisons, 3 attributs, 11 indices).
// Etape 1 : definissez groups_5 (Prenom, Boisson, Loisir), domaine 0..4 + 3 Distinct.
// Etape 2 : encodez les 11 indices de l'enonce (egalites, adjacences, "a gauche", "immediatement a gauche").
// Etape 3 : check() -> sat, puis compter_solutions(s, v) -> attendu exactement 1.

// TODO etudiant : groups_5 + solveur avec domaine 0..4 + 3 Distinct + les 11 indices
fn resoudre_cinq_amies() -> Option<(Solver, HashMap<String, Int>)> {
    None // TODO etudiant
}

println!("Exercice a completer : l'enigme des cinq amies (5 maisons, 11 indices).");

Exercice a completer : l'enigme des cinq amies (5 maisons, 11 indices).


## Conclusion

L'enigme d'Einstein illustre le gain du **declaratif** sur une classe de problemes ou l'approche humaine (deduction progressive sur un tableau) et l'approche naive (brute force) sont toutes deux laborieuses :

- L'**encodage par position** (une variable entiere = position de chaque valeur) transforme un puzzle apparemment qualitatif en un système de contraintes arithmetiques : egalites (`Brit == Red`), adjacences (`|Blend - Cat| == 1`), permutations (`Ast::distinct`).
- La **propagation de contraintes** resout les 25 variables entrelacees en millisecondes la ou la brute force affronte ~25 milliards de combinaisons ; le solveur **prouve l'unicite** en niant le temoin et en observant `unsat`.
- Le modèle est **paramètre par les données** : ajouter un indice, un attribut ou une maison se fait en modifiant l'instance, sans re-ecrire l'algorithme de resolution.

**Contraste avec le job-shop (notebook 08)** : ici `Solver` suffit (on cherche une solution realisable, pas un optimum) - c'est la **satisfiabilite** pure, la ou l'ordonnancement relevait de l'**optimisation** (`Optimize.minimize` du makespan). Les deux postures du solveur, posees au notebook 01, trouvent ici leur illustration complementaire.

Suite logique : retour a la [serie Z3-Python](README.md), ou explorez la version C# [Z3.Linq](../Z3-Linq2Z3/README.md) qui aborde la même enigme via la couche declarative LINQ.